# Neural Language Modeling

## Project Overview

This notebook explores neural language modeling using text data.
It covers data loading, vocabulary construction, text preprocessing,
model development, training, and evaluation.

####  Import Libraries

In [1]:
import io, sys, math, re
from collections import defaultdict
import numpy as np

In [3]:
# from google.colab import drive
# drive.mount('/content/drive'

## 3. Data Loading and Vocabulary Construction

This section defines a function to read tokenized text data,
organize sentences into lists of words, and construct a
vocabulary containing word-frequency counts.

In [5]:
# data_loader
def load_data(filename):
    '''
    parameters:
    filename (string): datafile
    
    Returns:
    data (list of lists): each list is a sentence of the text 
    vocab (dictionary): {word: no of times it appears in the text}
    '''
    fin = io.open(filename, 'r', encoding='utf-8')
    data = []
    vocab = defaultdict(lambda:0)
    for line in fin:
        sentence = line.split()
        data.append(sentence)
        for word in sentence:
            vocab[word] += 1
    return data, vocab

### Load Training and Validation Datasets

In [7]:

print("load training set..")
print("\n")
train_data, vocab = load_data("train1.txt")
print(train_data[0] if train_data else "No training sentences loaded")
print("\n")
print("how :",vocab['how'])
print("load validation set")
valid_data, _ = load_data("valid1.txt")


load training set..


['<s>', 'my', 'fathers', "don't", 'speak', 'dutch.', '</s>']


how : 107
load validation set


#### Text Preprocessing

Rare words are replaced with the `<unk>` token to handle
infrequent vocabulary items and reduce vocabulary sparsity.
The special sentence-boundary tokens `<s>` and `</s>` are preserved.

In [9]:
def remove_rare_words(data, vocab, mincount=1):
    """Replace unseen words and words occurring at most mincount times with <unk>."""
    return [[word if word in ("<s>", "</s>") or vocab.get(word, 0) > mincount else "<unk>"
             for word in sentence] for sentence in data]

#### Handling Rare Words in Training and Validation Data

The preprocessing function is applied to both datasets using
word frequencies calculated from the training set.

In [11]:
print("remove rare words")
train_data = remove_rare_words(train_data, vocab, mincount = 1)
valid_data = remove_rare_words(valid_data, vocab, mincount = 1)
#train_data
print(train_data[0] if train_data else "No training sentences loaded")


remove rare words
['<s>', 'my', '<unk>', "don't", 'speak', '<unk>', '</s>']


## N-Gram Language Model Construction

This section implements an n-gram language model that estimates the probability of a word based on its preceding context. The model counts word occurrences within different contexts and converts these counts into conditional probabilities using maximum likelihood estimation (MLE).

For a word \(w_i\) and its preceding context, the conditional probability is estimated as:

$$
P(w_i \mid w_{i-n+1}, \ldots, w_{i-1})
=
\frac{C(w_{i-n+1}, \ldots, w_i)}
{C(w_{i-n+1}, \ldots, w_{i-1})}
$$

where \(C\) denotes the frequency of a word sequence in the training corpus.

In [13]:
def build_ngram(data, n):
    '''
    Parameters:
    data (list of lists): each list is a sentence of the text 
    n (int): size of the n-gram
    
    Returns:
    proba (dictionary of dictionary)
    {
        context: {word:probability of this word given context}
    }
    
    
    '''
    total_number_words = 0
    counts = defaultdict(lambda: defaultdict(lambda: 0.0))

    for sentence in data:
      sentence = tuple(sentence)
      # dict can be indexed by tuples
      # store in the same dict all the ngrams
      # by using the context as a key and the word as a value
      for i in range(len(sentence)):
        total_number_words +=1
        for k in range(n):
          if i-k < 0:
            break
          counts[sentence[i-k:i]][sentence[i]] +=1 
                        

    proba  = defaultdict(lambda: defaultdict(lambda: 0.0))
    # Build the probabilities from the counts
    # Be careful with how you normalize!

    for context in counts.keys():
      denom =0
      for w in counts[context].keys():
            denom += counts[context][w]
      for w in counts[context].keys():
        proba[context][w] = counts[context][w]/denom 
    
    return proba

#### Illustrating N-Gram Generation

In [15]:
b = ["the", "boy", "is", "very", "good", "at", "footbal"]
n = 3

ngram_list =[]
for i in range(len(b)-n+1):
  start = i
  end = i + n
  ngram_list.append(b[start: end])
ngram_list

[['the', 'boy', 'is'],
 ['boy', 'is', 'very'],
 ['is', 'very', 'good'],
 ['very', 'good', 'at'],
 ['good', 'at', 'footbal']]

#### Building the N-Gram Language Model

In [17]:
# RUN TO BUILD NGRAM MODEL
n = 2
print("build ngram model with n = ", n)
model = build_ngram(train_data, n)

build ngram model with n =  2


##### Backoff Probability Estimation
Here, implement a recursive function over shorter and shorter context to compute a "stupid backoff model". An interpolation model can also be implemented this way.

In [20]:
def get_prob(model, context, w):
    """Back off to shorter contexts, discounting each backoff by 0.4."""
    context = tuple(context)
    if context in model and w in model[context]:
        return model[context][w]
    if not context:
        # An unknown word must be mapped to <unk> in preprocessing.
        return model.get((), {}).get("<unk>", 0.0) if w != "<unk>" else 0.0
    return 0.4 * get_prob(model, context[1:], w)

#### Model Evaluation Using Perplexity

In [22]:
def perplexity(model, data, n):
    """Return perplexity; infinity indicates an impossible token under the model."""
    perp, T = 0.0, 0
    for sentence in data:
        sentence = tuple(sentence)
        for i in range(1, len(sentence)):
            nc = min(n - 1, i)
            context = sentence[i-nc:i] if nc else ()
            prob = get_prob(model, context, sentence[i])
            if prob <= 0:
                return float("inf")
            perp -= math.log(prob)
            T += 1
    return math.exp(perp / T) if T else float("nan")

#### Evaluating the Model on the Validation Set

In [24]:
# COMPUTE PERPLEXITY ON VALIDATION SET

print("The perplexity is", perplexity(model, valid_data, n=n))

The perplexity is 53.582886955857006


## Next-Word Probability Distribution

This section implements a function to retrieve the probability distribution of possible next words given a context.

In [26]:
def get_proba_distrib(model, context):
    """Return the next-word distribution from the longest available context."""
    context = tuple(context)
    while context not in model or not model[context]:
        if not context:
            return {}
        context = context[1:]
    return dict(model[context])

## Sentence Generation

The trained n-gram language model is used to generate new sentences through probabilistic sampling.

In [28]:
def generate(model):
    """Sample a sentence from <s> to </s> (or stop after 100 tokens)."""
    sentence = ["<s>"]
    max_context = max((len(context) for context in model), default=0)
    while sentence[-1] != "</s>" and len(sentence) < 100:
        context = tuple(sentence[-max_context:]) if max_context else ()
        distribution = get_proba_distrib(model, context)
        if not distribution:
            break
        words = list(distribution)
        probabilities = np.array(list(distribution.values()), dtype=float)
        probabilities /= probabilities.sum()
        sentence.append(str(np.random.choice(words, p=probabilities)))
    return sentence

In [30]:
# GENERATE A SENTENCE FROM THE MODEL

print("Generated sentence: ",generate(model))

Generated sentence:  ['<s>', 'tom', 'and', 'mary', 'looked', 'up', 'from', 'the', 'separatists', "aren't", '<unk>', 'himself', 'to', '<unk>', '</s>']
